<h1 style="text-align:center;">37th International Summer School of the Swiss Association of Actuaries on</h1>
<h4 style="text-align:center;">“Insurance: Innovations for Products, Sustainability and Regulation”,</h4>
<p style="text-align:center;">Lausanne, 10.-14.08.2026</p>


# Day 2

## Overview

1) **Continuous trading vs. Buy-And-Hold strategy**  
Use the constant log-optimal portfolio process $\pi_t = \frac{b-r}{\sigma^2}$ and compare its wealth process with a buy-and-hold strategy that initially invests $x\frac{b-r}{\sigma^2}$ in the stock and $x\left(1-\frac{b-r}{\sigma^2}\right)$ in the money market account. Use the same stock-price paths and compare the maturities $T\in\{0.1,1,5\}$.

2) **CPPI strategy**  
Illustrate the wealth process of a CPPI strategy. Examine gap risk using large multipliers and rough time discretizations, and show how the risk nearly disappears when trading becomes sufficiently frequent.

3) **Log-optimal strategy vs. value-preserving strategy (Bonus)**  
Implement the log-optimal portfolio and consumption strategy and the value-preserving strategy. Compare both under varying market coefficients and maturity $T$.


In [ ]:
import numpy as np

from utils.common import *
from utils.viz import *
from utils.simulators import *

## Exercise 1

### Continuous rebalancing versus buy-and-hold

We work in the one-stock Black-Scholes market

$$\mathrm{d}B_t=rB_t\,\mathrm{d}t, \qquad \mathrm{d}S_t=bS_t\,\mathrm{d}t+\sigma S_t\,\mathrm{d}W_t, \qquad B_0=1, \quad S_0>0.$$

The exact price processes are

$$B_t=\mathrm{e}^{rt}, \qquad S_t=S_0\exp\left[\left(b-\frac{1}{2}\sigma^2\right)t+\sigma W_t\right].$$

As we already saw yesterday, the growth-optimal fraction invested in the stock is

$$\pi^\ast=\frac{b-r}{\sigma^2}.$$

This is a **target fraction**, not a fixed number of shares. Keeping it constant requires continuous rebalancing between the stock and the money market account.

### The two wealth processes

The continuously rebalanced strategy satisfies

$$\frac{\mathrm{d}X_t^{\mathrm{cont}}}{X_t^{\mathrm{cont}}}=\left[r+\pi^\ast(b-r)\right]\mathrm{d}t+\pi^\ast\sigma\,\mathrm{d}W_t,$$

and therefore

$$X_t^{\mathrm{cont}}=x\exp\left[\left(r+\pi^\ast(b-r)-\frac{1}{2}(\pi^\ast)^2\sigma^2\right)t+\pi^\ast\sigma W_t\right].$$

The buy-and-hold investor chooses the same allocation at time zero but never trades again. Since $B_0=1$, the initial holdings are

$$\varphi_0=x(1-\pi^\ast), \qquad \varphi_1=\frac{x\pi^\ast}{S_0},$$

so buy-and-hold wealth is

$$X_t^{\mathrm{BH}}=\varphi_0B_t+\varphi_1S_t=x(1-\pi^\ast)B_t+x\pi^\ast\frac{S_t}{S_0}.$$

Its actual risky fraction evolves as

$$\pi_t^{\mathrm{BH}}=\frac{\varphi_1S_t}{X_t^{\mathrm{BH}}}$$

which is distributed according to a logit-normal distribution and satisfies the SDE

$$\mathrm{d}\pi_t^{\mathrm{BH}}=\pi_t^{\mathrm{BH}}(1-\pi_t^{\mathrm{BH}})\left[(b-r-\sigma^2\pi_t^{\mathrm{BH}})\,\mathrm{d}t+\sigma\,\mathrm{d}W_t\right] .$$

Note that this is a **mean-reverting process** of third order with long-term mean $\pi^\ast$. 

Both strategies start with the same wealth and the same stock exposure, but only the continuously rebalanced strategy keeps the fraction equal to $\pi^\ast$. The buy-and-hold strategy will slowly drift away from the target fraction.

In [ ]:
def simulate_continuous_vs_buy_and_hold(initial_wealth: float, initial_stock_price: float, market_params: FinancialMarket, sde_params: SDESimulationParameters) -> SDEOutput:
    '''Simulate both strategies under exactly the same stock-price paths.'''

    r = market_params.risk_free_rate
    sigma = market_params.volatility
    pi_star = market_params.risk_premium / sigma**2

    out = simulate_market_exact(initial_stock_price, market_params, sde_params)
    t = out.time_grid
    W = out.paths["driver"]
    B = out.paths["bond"]
    S = out.paths["stock"]

    continuous_wealth = initial_wealth * np.exp(
        (r + pi_star * market_params.risk_premium - 0.5 * pi_star**2 * sigma**2) * t[:, np.newaxis] + pi_star * sigma * W
    )

    bond_units  = initial_wealth * (1 - pi_star)
    stock_units = initial_wealth * pi_star / initial_stock_price

    buy_and_hold_wealth   = bond_units * B + stock_units * S
    buy_and_hold_fraction = stock_units * S / buy_and_hold_wealth

    return SDEOutput(
        time_grid=t,
        paths={
            "continuous_rebalancing": continuous_wealth,
            "buy_and_hold": buy_and_hold_wealth,
            "buy_and_hold_fraction": buy_and_hold_fraction,
            "driver": W
        },
    )


### Numerical validation

The simulation should satisfy three immediate checks. Both strategies start at $x$, both initially invest the fraction $\pi^\ast$ in the stock, and the Monte Carlo mean of log wealth under continuous rebalancing should be close to

$$\mathbb{E}[\log X_T^{\mathrm{cont}}]=\log x+\left[r+\pi^\ast(b-r)-\frac{1}{2}(\pi^\ast)^2\sigma^2\right]T.$$


In [ ]:
market = FinancialMarket(risk_free_rate=0.025, risk_premium=0.08, volatility=0.25)
params = SDESimulationParameters(time_horizon=1.0, time_steps=250, num_paths=1_000)

comparison = simulate_continuous_vs_buy_and_hold(250, 100, market, params) # We use X0 = 250, S0 = 100
x          = comparison.paths["continuous_rebalancing"][0, 0]  # initial wealth
pi_star    = market.risk_premium / market.volatility**2

expected_log_wealth = np.log(x) + (
    market.risk_free_rate + pi_star * market.risk_premium - 0.5 * pi_star**2 * market.volatility**2
) * params.time_horizon

check_condition(
    "Both strategies start at the same wealth",
    np.allclose(comparison.paths["continuous_rebalancing"][0, :], x)
    and np.allclose(comparison.paths["buy_and_hold"][0, :], x),
)
check_value(
    "Initial buy-and-hold stock fraction",
    comparison.paths["buy_and_hold_fraction"][0, 0],
    pi_star,
)
check_value(
    "Mean log wealth under continuous rebalancing",
    np.mean(np.log(comparison.paths["continuous_rebalancing"][-1, :])),
    expected_log_wealth,
    abs_tol=1e-2,
)


### Interactive comparison

The first panel compares the two strategies on the same market path. The second shows that the buy-and-hold fraction drifts away from/oscillates around $\pi^\ast$. The final panel estimates the paired difference

$$\mathbb{E}\left[\log X_T^{\mathrm{cont}}-\log X_T^{\mathrm{BH}}\right] = \mathbb{E}\left[\log\frac{X_T^{\mathrm{cont}}}{X_T^{\mathrm{BH}}}\right]$$

at different maturities. Here, one would expect the difference to be positive, since the continuously rebalanced strategy is growth-optimal. Moreover, note that we restrict the sliders to parameter combinations with $0 \leq \pi^\ast \leq 1$ to ensure positive buy-and-hold wealth and therefore a well-defined logarithm. 


In [ ]:
explore_continuous_vs_buy_and_hold(simulate_continuous_vs_buy_and_hold)


### Interpretation

- The two strategies are almost indistinguishable over short horizons because the buy-and-hold weight has little time to drift.
- A favorable stock move increases the buy-and-hold stock fraction, while an unfavorable move decreases it. Continuous rebalancing instead sells after relative stock gains and buys after relative stock losses to restore $\pi^\ast$.
- The continuously rebalanced strategy does not dominate on every path. Buy-and-hold can finish ahead in a particular scenario.
- The relevant log-utility statement is an expectation: continuous rebalancing maximizes $\mathbb{E}[\log X_T]$. The expected loss from not rebalancing is typically small near the optimum, which is the practical robustness point emphasized in Lecture 3A.


## Exercise 2

### CPPI and the continuous-time guarantee

Let $F$ denote the guarantee at maturity $T$. Its time-$t$ present value is the floor

$$F(t)=F\mathrm{e}^{-r(T-t)}.$$

For wealth $X_t$, define the cushion by

$$C_t=X_t-F(t).$$

A CPPI strategy with multiplier $M$ invests the monetary amount

$$A_t=MC_t$$

in the stock and the remainder in the money market account. Under continuous trading, the cushion satisfies

$$\frac{\mathrm{d}C_t}{C_t} = \left( Mb + (1-M)r \right)\mathrm{d}t + M\sigma \,\mathrm{d}W_t .$$

This is a geometric Brownian motion. Consequently,

$$C_t=C_0\exp\left[\left(r+M(b-r)-\frac{1}{2}M^2\sigma^2\right)t+M\sigma W_t\right], \qquad C_0=x-F\mathrm{e}^{-rT},$$

and $X_t=F(t)+C_t>F(t)$ whenever $C_0>0$. In the ideal continuous-time Black-Scholes model, the guarantee is therefore respected.

### Where gap risk enters

In practice, the portfolio is rebalanced only at dates $t_n$. At time $t_n$, the discrete CPPI chooses

$$A_n=M\max\left(X_{t_n}-F(t_n),0\right).$$

It then keeps this monetary allocation unchanged until $t_{n+1}$. Using exact stock and bond returns over the interval gives

$$X_{t_{n+1}}=A_n\frac{S_{t_{n+1}}}{S_{t_n}}+\left(X_{t_n}-A_n\right)\mathrm{e}^{r\Delta t}.$$

A sufficiently large stock loss can make $X_{t_{n+1}}<F(t_{n+1})$. This is the so-called *gap risk*. It becomes more severe for a large multiplier $M$ and a long interval $\Delta t$. The optional cap $A_n\leq X_{t_n}$ follows the practical safeguard mentioned in the slides and prevents borrowing to finance the stock position.


In [ ]:
def simulate_cppi(initial_wealth: float, guarantee: float, multiplier: float, cap_at_wealth: bool, market_params: FinancialMarket, sde_params: SDESimulationParameters) -> SDEOutput:
    '''Simulate continuous-time and discretely rebalanced CPPI paths.'''

    if multiplier < 0:
        raise ValueError("The CPPI multiplier must be non-negative.")

    r = market_params.risk_free_rate
    b = market_params.drift
    sigma = market_params.volatility
    T = sde_params.time_horizon
    dt = sde_params.dt

    out = simulate_market_exact(100, market_params, sde_params)
    t = out.time_grid
    W = out.paths["driver"]
    S = out.paths["stock"]

    stock_gross_return  = S[1:, :] / S[:-1, :]
    bond_gross_return   = np.exp(r * dt)

    floor = guarantee * np.exp(-r * (T - t))[:, np.newaxis]
    initial_cushion = initial_wealth - floor[0, 0]

    if initial_cushion <= 0:
        raise ValueError("Initial wealth must be strictly above the initial floor.")

    # Implementing cushion dynamics
    # C_t = C_0 * exp((r + M*(b - r) - 0.5*M^2*sigma^2)*t + M*sigma*W_t)
    continuous_cushion = initial_cushion * np.exp(
        (r + multiplier * market_params.risk_premium - 0.5 * multiplier**2 * sigma**2) * t[:, np.newaxis]
        + multiplier * sigma * W
    )
    continuous_wealth = floor + continuous_cushion

    discrete_wealth = np.zeros_like(S) # X_t
    risky_amount    = np.zeros_like(S) # A_t
    cap_active      = np.zeros_like(S)
    discrete_wealth[0, :] = initial_wealth
    for step in range(sde_params.time_steps):
        cushion = np.maximum(discrete_wealth[step, :] - floor[step, 0], 0.0)
        target_risky_amount = multiplier * cushion

        if cap_at_wealth:
            target_risky_amount = np.minimum(target_risky_amount, np.maximum(discrete_wealth[step, :], 0.0))

            mask = target_risky_amount >= discrete_wealth[step, :]
            cap_active[step, mask] = 1

        risky_amount[step, :] = target_risky_amount
        safe_amount           = discrete_wealth[step, :] - target_risky_amount

        # Updating wealth
        # X_{n+1} = A_n * (S_{n+1}/S_n) + (X_n - A_n)exp(r * Delta t)
        discrete_wealth[step + 1, :] = target_risky_amount * stock_gross_return[step, :] + safe_amount * bond_gross_return

    risky_amount[-1, :] = risky_amount[-2, :]

    return SDEOutput(
        time_grid=t,
        paths={
            "stock": S,
            "floor": np.broadcast_to(floor, S.shape),
            "continuous_cppi": continuous_wealth,
            "discrete_cppi": discrete_wealth,
            "risky_amount": risky_amount,
            "cap_active": cap_active,
        },
    )

### Numerical validation

The continuous-time cushion is an exponential process and must remain strictly positive. The discrete strategy, by contrast, may breach the floor. The next cell checks the continuous guarantee and reports the estimated discrete gap probability for monthly rebalancing.


In [ ]:
market = FinancialMarket(risk_free_rate=0.02, risk_premium=0.06, volatility=0.25)
params = SDESimulationParameters(time_horizon=1.0, time_steps=12, num_paths=5_000)

cppi = simulate_cppi(
    initial_wealth=250.0,
    guarantee=75.0,
    multiplier=5.0,
    cap_at_wealth=False,
    market_params=market,
    sde_params=params,
)

continuous_cushion = cppi.paths["continuous_cppi"] - cppi.paths["floor"]
discrete_gap = np.any(cppi.paths["discrete_cppi"] < cppi.paths["floor"] - 1e-10, axis=0)

check_condition("Continuous-time CPPI stays strictly above the floor", np.all(continuous_cushion > 0))
print(f"Estimated monthly-rebalancing gap probability: {np.mean(discrete_gap):.2%}")


### Interactive gap-risk experiment

Use the multiplier and trading-frequency controls to search for floor breaches. The path panel automatically selects the most stressed path in the current simulation, making a breach visible whenever one occurs. The central panel shows the distribution of the final wealth and the amount we wanted to guarantee. For large values of $M$ and low trading frequencies, we expect a non-negligible part of the histogram to fall below the guarantee, whereas the continuous time-model should always stay above it. The right panel repeats the experiment across trading frequencies. Since it is a Monte Carlo estimate, the curve need not be perfectly monotone, but the broad pattern should be clear.


In [ ]:
explore_cppi_gap_risk(simulate_cppi)

### Interpretation

- In the continuous-time model, the cushion is lognormal and cannot cross zero, so terminal wealth remains above the guarantee.
- In discrete time, the risky amount is frozen between trading dates. A stock loss larger than the cushion can therefore create a shortfall before the strategy can rebalance.
- A larger multiplier increases both upside participation and gap risk.
- More frequent rebalancing shortens the period over which the allocation is frozen, so the estimated gap probability becomes very small for fine grids.
- Capping the stock investment at total wealth removes CPPI borrowing. It substantially reduces extreme losses, although it also limits upside exposure and does not turn a discretely monitored strategy into a mathematical continuous-time guarantee.


## Exercise 3 (Bonus)

### Log-optimal versus value-preserving strategies

This is a generalization of the third exercise. Whereas before we restricted ourselves to $T = 1$, we will now derive the log-optimal portfolio and consumption strategy for arbitrary maturities $T>0$. 

We use the one-stock market price of risk

$$\theta=\frac{b-r}{\sigma}$$

and the state-price density

$$H_t=\exp\left[-\left(r+\frac{1}{2}\theta^2\right)t-\theta W_t\right].$$

For the objective

$$\max_{(\pi,c)}\mathbb{E}\left[\log X_T+\int_0^T\log c_t\,\mathrm{d}t\right],$$

the inverse marginal utility is $I(y)=\frac{1}{y}$. The budget equation gives the multiplier $y=\frac{T+1}{x}$ and therefore

$$c_t^{\log}=\frac{x}{(T+1)H_t}, \qquad X_t^{\log}=\frac{x(T-t+1)}{(T+1)H_t}, \qquad \pi_t^{\log}=\frac{b-r}{\sigma^2}.$$

The consumption-to-wealth ratio is deterministic:

$$\frac{c_t^{\log}}{X_t^{\log}}=\frac{1}{T-t+1}.$$

Setting $T = 1$, we see that these results coincide with yesterday's.

### The value-preserving counterpart

The one-stock value-preserving strategy from Lecture 6 keeps its portfolio value on the deterministic path

$$X_t^{\mathrm{VPS}}=xB_t=x\mathrm{e}^{rt}.$$

Its holdings and cumulative consumption satisfy

$$\varphi_0(t)=x\left(1-\frac{b-r}{\sigma^2}\right), \qquad \varphi_1(t)=x\frac{b-r}{\sigma^2}\frac{B_t}{S_t},$$

and

$$\mathrm{d}C_t^{\mathrm{VPS}}=xB_t\left(\theta^2\,\mathrm{d}t+\theta\,\mathrm{d}W_t\right).$$

The value-preserving strategy uses the same risky fraction $\frac{b-r}{\sigma^2}$ as the growth-optimal portfolio, but it continuously extracts or injects the gains needed to keep wealth equal to $xB_t$. In particular, $C_t^{\mathrm{VPS}}$ need not be increasing on every path; negative increments represent capital injections rather than ordinary nonnegative consumption.


In [ ]:
def simulate_log_vs_value_preserving(initial_wealth: float, market_params: FinancialMarket, sde_params: SDESimulationParameters) -> SDEOutput:
    '''Simulate both strategies under a common Brownian driver.'''

    out = simulate_brownian_path(params=sde_params)
    t   = out.time_grid
    dW  = out.paths["dW"]
    W   = out.paths["W"]

    x = initial_wealth
    r = market_params.risk_free_rate
    theta = market_params.risk_premium / market_params.volatility
    T = sde_params.time_horizon
    dt = sde_params.dt

    bond = np.exp(r * t)[:, np.newaxis]
    H = np.exp(-(r + 0.5 * theta**2) * t[:, np.newaxis] - theta * W) # Again, our deflator from yesterday

    # log-optimal consumption and wealth
    log_consumption_rate       = x / ((T + 1) * H)
    log_wealth                 = x * (T - t[:, np.newaxis] + 1) / ((T + 1) * H)
    log_cumulative_consumption = cumulative_trapezoid(log_consumption_rate, dt)

    # value-preserving optimal consumption and wealth
    vps_wealth = np.broadcast_to(x * bond, H.shape)
    vps_consumption_increment = x * bond[:-1, :] * (theta**2 * dt + theta * dW)

    vps_cumulative_consumption = np.zeros_like(H)
    vps_cumulative_consumption[1:, :] = np.cumsum(vps_consumption_increment, axis=0)

    return SDEOutput(
        time_grid=t,
        paths={
            "deflator": H,
            "log_wealth": log_wealth,
            "log_consumption_rate": log_consumption_rate,
            "log_cumulative_consumption": log_cumulative_consumption,
            "vps_wealth": vps_wealth,
            "vps_cumulative_consumption": vps_cumulative_consumption,
        },
    )

### Analytical expectations

Define the market price of risk and the abbreviation

$$\theta=\frac{b-r}{\sigma}, \qquad k=r+\theta^2,$$

and let

$$\mathcal{E}^z(x):=\begin{cases}\frac{\mathrm{e}^{zx}-1}{z}, & z\neq 0,\\ x, & z=0.\end{cases}$$

The state-price density satisfies

$$H_t^{-1}=\exp\left[\left(r+\frac{1}{2}\theta^2\right)t+\theta W_t\right].$$

Using $\mathbb{E}[\mathrm{e}^{\theta W_t}]=\mathrm{e}^{\frac{1}{2}\theta^2t}$ therefore gives

$$\mathbb{E}[H_t^{-1}]=\mathrm{e}^{(r+\theta^2)t}=\mathrm{e}^{kt}.$$

For the log-optimal strategy,

$$X_T^{\log}=\frac{x}{(T+1)H_T}, \qquad c_t^{\log}=\frac{x}{(T+1)H_t}, \qquad C_T^{\log}=\int_0^T c_t^{\log}\,\mathrm{d}t.$$

Consequently,

$$\mathbb{E}[X_T^{\log}]=\frac{x}{T+1}\mathbb{E}[H_T^{-1}]=\frac{x}{T+1}\mathrm{e}^{kT},$$

and, by interchanging expectation and integration,

$$\mathbb{E}[C_T^{\log}]=\frac{x}{T+1}\int_0^T\mathbb{E}[H_t^{-1}]\,\mathrm{d}t=\frac{x}{T+1}\int_0^T\mathrm{e}^{kt}\,\mathrm{d}t=\frac{x}{T+1}\mathcal{E}^k(T).$$

For the value-preserving strategy,

$$X_t^{\mathrm{VPS}}=xB_t=x\mathrm{e}^{rt}, \qquad \mathrm{d}C_t^{\mathrm{VPS}}=xB_t\left(\theta^2\,\mathrm{d}t+\theta\,\mathrm{d}W_t\right), \qquad C_0^{\mathrm{VPS}}=0.$$

Since the stochastic integral has expectation zero,

$$\mathbb{E}[C_T^{\mathrm{VPS}}]=x\theta^2\int_0^T\mathrm{e}^{rt}\,\mathrm{d}t=x\theta^2\mathcal{E}^r(T).$$

Thus, the four expected values are

$$\boxed{\begin{aligned}\mathbb{E}[X_T^{\log}]&=\frac{x}{T+1}\mathrm{e}^{kT}, & \qquad \mathbb{E}[C_T^{\log}]&=\frac{x}{T+1}\mathcal{E}^k(T),\\ \mathbb{E}[X_T^{\mathrm{VPS}}]&=x\mathrm{e}^{rT}, & \qquad \mathbb{E}[C_T^{\mathrm{VPS}}]&=x\theta^2\mathcal{E}^r(T).\end{aligned}}$$

In [ ]:
x = 1.0
T = 2.5
market = FinancialMarket(risk_free_rate=0.025, risk_premium=0.05, volatility=0.25)
params = SDESimulationParameters(time_horizon=T, time_steps=int(250*T), num_paths=15_000)

strategies = simulate_log_vs_value_preserving(x, market, params)

theta = market.risk_premium / market.volatility
k     = market.risk_free_rate + theta**2

# Exact expected values for the log-optimal and value-preserving strategies
expected_log_wealth      = x * np.exp(k * T) / (T + 1)
expected_log_consumption = x * exponential_integral(k, T) / (T + 1)
expected_vps_wealth      = x * np.exp(market.risk_free_rate * T)
expected_vps_consumption = x * theta**2 * exponential_integral(market.risk_free_rate, T)

check_value(
    "Mean log-optimal terminal wealth",
    np.mean(strategies.paths["log_wealth"][-1, :]),
    expected_log_wealth,
    abs_tol=1e-3,
    rel_tol=1e-6,
)
check_value(
    "Mean log-optimal cumulative consumption",
    np.mean(strategies.paths["log_cumulative_consumption"][-1, :]),
    expected_log_consumption,
    abs_tol=1e-3,
    rel_tol=1e-6,
)
check_value(
    "Value-preserving terminal wealth",
    strategies.paths["vps_wealth"][-1, 0],
    expected_vps_wealth,
    abs_tol=1e-3,
    rel_tol=1e-6,
)
check_value(
    "Mean value-preserving cumulative consumption",
    np.mean(strategies.paths["vps_cumulative_consumption"][-1, :]),
    expected_vps_consumption,
    abs_tol=1e-3,
    rel_tol=1e-6,
)

### Interactive comparison

The first two panels use the same Brownian path for both strategies. The final panel compares Monte Carlo means with their analytical values. Vary $r$, $b$, $\sigma$, and $T$ to see how the market price of risk $\theta$ affects the split between wealth and consumption.


In [ ]:
explore_log_vs_value_preserving(simulate_log_vs_value_preserving)


### Interpretation

- The log-optimal investor lets wealth respond to the market and consumes at a strictly positive state-dependent rate. As maturity approaches, the fraction of wealth consumed per unit of time increases.
- The value-preserving strategy keeps wealth on the risk-free path $xB_t$. Market gains are removed as cumulative consumption, while adverse shocks can require negative consumption increments, i.e. injections.
- Both strategies use the same growth-optimal risky fraction, but they apply different rules to the gains generated by that exposure.
- A larger market price of risk $|\theta|$ makes both consumption processes more variable and increases their expected growth under the physical measure.
- Increasing $T$ makes the different intertemporal objectives especially visible: the log-optimal strategy shifts more of its initial state-price budget toward consumption, while the value-preserving strategy protects the deterministic portfolio-value path.